# Univariate Linear Regression
## Using Snowpark Python and Scikit-Learn
### Overview
This notebook builds and evaluates a linear model to predict housing prices using a single input feature.

Steps:
- Setup
- Load and Explore Data
- Prepare Data for Regression
- Train and Examine Linear Model
- Evaluate the Model
- Examine the Model Visually

### Setup

In [ ]:
import snowflake.snowpark
from snowflake.snowpark.session import Session

import seaborn as sns

# config_dir = '/home/jovyan/.ssh'
# configfile = config_dir + '/sf_config'

* Load configuration and connect to Snowflake

In [ ]:
# My code for Snowflake account connection

CONFIG_DIR = '/Users/richardkirk/.ssh'
CONFIGFILE = CONFIG_DIR + '/sf_config'


# Load configuration file
with open(CONFIGFILE) as f:
    lines = f.readlines()
    
# Convert configuration to a properties map
props = {}
for line in lines:
    (key, value) = line.split('=')
    props.update({key.lower() : value[0:-1]})
    
# Convert the private key to a DER-encoded bytes object
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

with open(props['private_key_file'], "rb") as key:
    private_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend()
    )
    
private_key_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

# Connect to Snowflake
session = Session.builder.configs({**props, **{"private_key": private_key_bytes}}).create()

### Load and Explore Data
* Define a Snowpark DataFrame on Snowflake table **usa_housing**.

In [ ]:
housingDF = session.table('data_science_db.housing.usa_housing')

* Examine the data.

In [ ]:
housingDF.schema.fields

In [ ]:
housingDF.count()

In [ ]:
housingDF.show(5)

- Load as a Pandas DataFrame and examine further

In [ ]:
housingPDF = housingDF.toPandas()

In [ ]:
housingPDF.head()

In [ ]:
housingPDF.info()

In [ ]:
housingPDF.describe()

In [ ]:
sns.pairplot(housingPDF)

In [ ]:
sns.histplot(housingPDF['PRICE'])

In [ ]:
sns.heatmap(housingPDF.corr())

*Interpretation:* Of the available features, AVERAGE_AREA_INCOME shows the highest correlation with PRICE. For an initial model, use that feature alone.

* Using Snowflake, compute the correlation between area income and price.

In [ ]:
housingDF.stat.corr('average_area_income', 'price')

- Using Python, plot the two variables

In [ ]:
(   housingDF
    .select('AVERAGE_AREA_INCOME', 'PRICE')
    .toPandas()
    .plot.scatter(x='AVERAGE_AREA_INCOME', y='PRICE')
)

### Prepare Data for Regression

In [ ]:
# Select only the feature column and the target
housingDF = housingDF.select('average_area_income', 'price')

# Create train and test sets
(housing_trainDF, housing_testDF) = housingDF.random_split([0.8, 0.2], seed=42)

# Get train and test set sizes
(housing_trainDF.count(), housing_testDF.count())

- Get Pandas DataFrames for training in Scikit-Learn

In [ ]:
train_x_PDF = housing_trainDF.drop('price').toPandas()
train_y_PDF = housing_trainDF.select('price').toPandas()

### Train and Examine Linear Model
- Fit the model to the training data

In [ ]:
from sklearn.linear_model import LinearRegression
lin_reg = LinearRegression()
lin_reg.fit(train_x_PDF, train_y_PDF)

- Examine the model parameters

In [ ]:
print('Model intercept: ', lin_reg.intercept_[0])

In [ ]:
print('Model coefficent: ', lin_reg.coef_[0][0])

### Evaluate the Model
- Predict results for the test set

In [ ]:
# Fetch test data as Pandas DataFrames
test_x_PDF = housing_testDF.drop('price').toPandas()
test_y_PDF = housing_testDF.select('price').toPandas()

# Run predictions
predictions = lin_reg.predict(test_x_PDF)

- Display a few predictions

In [ ]:
import pandas as pd
predictions_PDF = pd.DataFrame(predictions, columns=['prediction'])
test_y_PDF.join(predictions_PDF)[:5]

- Calculate evaluation metrics 

**Record these metrics for comparison with another model.**

In [ ]:
from sklearn import metrics
print('r2:\t', metrics.r2_score(test_y_PDF, predictions))

In [ ]:
import math
print('rmse:\t', round(math.sqrt(metrics.mean_squared_error(test_y_PDF, predictions)), 2))

### Examine the Model Visually

- Scatter plot of predictions vs values

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(test_x_PDF, test_y_PDF, color='blue')
plt.plot(test_x_PDF, predictions, color='black')
plt.xlabel('Average Income')
plt.ylabel('Home Price')

- Distribution of prediction errors

In [ ]:
prediction_errors = test_y_PDF - predictions
prediction_errors = prediction_errors.rename(columns={"PRICE" : "Prediction Error"})
sns.displot(prediction_errors, bins=50)